# Pha S — Buoc 5: Doi chung doc lap thu 2 — Apnea-ECG (tim-ho hap)

**Muc tieu:** Fantasia (notebook 02c/03) da xac nhan TE(ho hap->tim) > TE(tim->ho hap) (RSA) bang KSG. De ket qua dang tin hon cho Q1/Q2, kiem tra lai tren 1 **bo du lieu DOC LAP THU 2**: Apnea-ECG, 8 ban ghi co kenh ho hap (`a01-a04, b01, c01-c03`).

**Khac Fantasia ve cau truc file** (da xac minh truc tiep): ECG va ho hap nam o **2 file WFDB rieng** cho cung 1 lan ghi - `a01` (chi ECG, co annotation QRS `.qrs` rieng) va `a01r` (chi ho hap: `Resp C`, `Resp A`, `Resp N`, `SpO2` - khong co annotation nhip). Dung kenh `Resp C` (chest - ho hap qua long nguc, tuong tu RESP cua Fantasia) lam kenh ho hap. Do dai 2 file lech nhau ~204 mau (~2s) tren tong ~8.2 gio - khong dang ke, `align_to_common_grid` xu ly an toan.

**Chi dung KSG** (Amortized da biet FAIL tren du lieu that - xem docs/PHASE_S_REPORT.md muc 4, lap lai o day khong co gia tri them). **Chi 8 ban ghi** nen suc manh thong ke han che - dung nhu 1 kiem tra HUONG NHAT QUAN, khong ky vong CI hep nhu Fantasia (23 [17 sau sua loc] ban ghi).

In [1]:
import warnings; warnings.filterwarnings('ignore')
import os
os.environ.setdefault('JAVA_HOME', r'C:\Program Files\Java\jdk-22')
import yaml, numpy as np, pandas as pd, matplotlib.pyplot as plt, wfdb
from pathlib import Path

from pqrst.data.real.cardiac import rr_from_beat_annotations, interpolate_rr
from pqrst.data.real.respiration import preprocess_respiration, bandpass_filter
from pqrst.data.real.sync import align_to_common_grid, sync_and_window, verify_synchronization
from pqrst.evaluation.sanity_check import bidirectional_te, run_sanity_check_per_record
from pqrst.baselines.ksg import KSGTEEstimator

BASE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
cfg = yaml.safe_load(open(BASE/'configs'/'real'/'preprocessing.yaml', encoding='utf-8'))
est_ksg = KSGTEEstimator()

RECORDS_WITH_RESP = ['a01', 'a02', 'a03', 'a04', 'b01', 'c01', 'c02', 'c03']

## 1. Chay pipeline cho 1 ban ghi mau (a01) - xem tung buoc

Giong cau truc notebook 02c (RR -> loc bang thong; RESP -> loc bang thong; dong bo; kiem tra dong bo BAT BUOC; cat cua so), khac o buoc doc du lieu (2 file).

In [2]:
record_id = 'a01'
ann = wfdb.rdann(str(BASE / 'data' / 'raw' / 'apnea-ecg' / record_id), 'qrs')
beat_times_raw = ann.sample / 100.0
is_normal = np.array([s == 'N' for s in ann.symbol])
beat_times = beat_times_raw[is_normal]
print(f'Tong nhip: {len(beat_times_raw)}, nhip N: {len(beat_times)}')

t_rr, rr = rr_from_beat_annotations(beat_times, exclude_ectopic=True,
                                     threshold_ratio=cfg['cardiac']['ectopic_threshold_ratio'])
t_grid_rr, rr_grid = interpolate_rr(t_rr, rr, grid_fs=cfg['grid_fs'])
rr_grid_filtered = bandpass_filter(rr_grid, cfg['grid_fs'], tuple(cfg['respiration']['bandpass_hz']))
print(f'RR grid: {len(rr_grid)} mau')

resp_record = wfdb.rdrecord(str(BASE / 'data' / 'raw' / 'apnea-ecg' / (record_id + 'r')))
print('Kenh ho hap co san:', resp_record.sig_name)
resp_sig = resp_record.p_signal[:, resp_record.sig_name.index('Resp C')]
t_resp, resp_grid = preprocess_respiration(
    resp_sig, fs=resp_record.fs, grid_fs=cfg['grid_fs'],
    bandpass=tuple(cfg['respiration']['bandpass_hz']))
print(f'RESP grid: {len(resp_grid)} mau')

Tong nhip: 29938, nhip N: 29938
RR grid: 118253 mau
Kenh ho hap co san: ['Resp C', 'Resp A', 'Resp N', 'SpO2']
RESP grid: 118272 mau


## 2. Dong bo + kiem tra dong bo (BAT BUOC)

In [3]:
rr_aligned, resp_aligned = align_to_common_grid(
    t_grid_rr, rr_grid_filtered, t_resp, resp_grid, grid_fs=cfg['grid_fs'])
print(f'Sau dong bo: {len(rr_aligned)} mau chung ({len(rr_aligned)/cfg["grid_fs"]/60:.1f} phut)')

sync_check = verify_synchronization(rr_aligned, resp_aligned, grid_fs=cfg['grid_fs'],
                                     estimator=est_ksg,
                                     shift_seconds=cfg['sync_check']['shift_seconds'],
                                     max_ratio_for_pass=cfg['sync_check']['max_ratio_for_pass'])
print(sync_check)

Sau dong bo: 118252 mau chung (492.7 phut)
{'te_aligned': 0.16290929862708686, 'te_shifted': 0.05675715596875316, 'ratio': 0.34839727656476555, 'passed': True, 'inconclusive': False}


## 3. Lap lai cho ca 8 ban ghi + kiem dinh CHINH THUC theo don vi ban ghi

In [4]:
def process_apnea_record(record_id: str) -> dict:
    report = {'record_id': record_id}
    try:
        ann = wfdb.rdann(str(BASE / 'data' / 'raw' / 'apnea-ecg' / record_id), 'qrs')
        beat_times_raw = ann.sample / 100.0
        is_normal = np.array([s == 'N' for s in ann.symbol])
        beat_times = beat_times_raw[is_normal]

        t_rr, rr = rr_from_beat_annotations(
            beat_times, exclude_ectopic=True,
            threshold_ratio=cfg['cardiac']['ectopic_threshold_ratio'])
        removed_frac = 1 - len(rr) / max(len(beat_times) - 1, 1)
        report['ectopic_removed_frac'] = float(removed_frac)
        if removed_frac > cfg['cardiac']['max_removed_fraction']:
            report['verdict'] = 'FAIL'
            report['reason'] = f'loai qua nhieu ectopic ({removed_frac*100:.1f}%)'
            return report

        t_grid_rr, rr_grid = interpolate_rr(t_rr, rr, grid_fs=cfg['grid_fs'])
        rr_grid_filtered = bandpass_filter(rr_grid, cfg['grid_fs'], tuple(cfg['respiration']['bandpass_hz']))

        resp_record = wfdb.rdrecord(str(BASE / 'data' / 'raw' / 'apnea-ecg' / (record_id + 'r')))
        resp_sig = resp_record.p_signal[:, resp_record.sig_name.index('Resp C')]
        t_resp, resp_grid = preprocess_respiration(
            resp_sig, fs=resp_record.fs, grid_fs=cfg['grid_fs'],
            bandpass=tuple(cfg['respiration']['bandpass_hz']))

        rr_aligned, resp_aligned = align_to_common_grid(
            t_grid_rr, rr_grid_filtered, t_resp, resp_grid, grid_fs=cfg['grid_fs'])
        report['n_aligned_samples'] = int(len(rr_aligned))

        sync_check = verify_synchronization(
            rr_aligned, resp_aligned, grid_fs=cfg['grid_fs'], estimator=est_ksg,
            shift_seconds=cfg['sync_check']['shift_seconds'],
            max_ratio_for_pass=cfg['sync_check']['max_ratio_for_pass'])
        report['sync_ratio'] = sync_check.get('ratio')
        if sync_check.get('inconclusive'):
            report['verdict'] = 'FAIL'
            report['reason'] = 'sync check inconclusive (te_aligned qua nho)'
            return report
        if not sync_check['passed']:
            report['verdict'] = 'FAIL'
            report['reason'] = f"sync check khong dat (ratio={sync_check['ratio']:.2f})"
            return report

        w_rr_to_resp = sync_and_window(rr_aligned, resp_aligned, grid_fs=cfg['grid_fs'],
                                       window_seconds=cfg['window_seconds'], record_id=record_id,
                                       config_name='rr_to_resp')
        w_resp_to_rr = sync_and_window(resp_aligned, rr_aligned, grid_fs=cfg['grid_fs'],
                                       window_seconds=cfg['window_seconds'], record_id=record_id,
                                       config_name='resp_to_rr')
        te_res = bidirectional_te(w_resp_to_rr, w_rr_to_resp, est_ksg)
        report['verdict'] = 'PASS'
        report['te_resp_to_rr'] = te_res['te_forward_mean']
        report['te_rr_to_resp'] = te_res['te_backward_mean']
        report['n_windows'] = te_res['n_windows']
        return report
    except Exception as e:
        report['verdict'] = 'ERROR'
        report['reason'] = f'{type(e).__name__}: {e}'
        return report


reports = []
for rid in RECORDS_WITH_RESP:
    r = process_apnea_record(rid)
    print(f"[{rid}] {r['verdict']}" + (f": {r.get('reason','')}" if r.get('reason') else ''))
    reports.append(r)

summary = pd.DataFrame(reports)
out_dir = BASE / 'data' / 'processed' / 'apnea-ecg'
out_dir.mkdir(parents=True, exist_ok=True)
summary.to_csv(out_dir / 'quality_report.csv', index=False)
summary

[a01] PASS
[a02] PASS
[a03] PASS
[a04] FAIL: sync check inconclusive (te_aligned qua nho)
[b01] PASS
[c01] PASS
[c02] FAIL: sync check khong dat (ratio=0.98)
[c03] PASS


,record_id,ectopic_removed_frac,n_aligned_samples,sync_ratio,verdict,te_resp_to_rr,te_rr_to_resp,n_windows,reason
0,a01,0.009353,118252,0.348397,PASS,0.117869,0.132115,985.0,NaN
1,a02,0.130949,127261,0.435907,PASS,0.039981,0.048434,1060.0,NaN
2,a03,0.019444,125378,0.587769,PASS,0.082758,0.073998,1044.0,NaN
3,a04,0.006212,119158,NaN,FAIL,NaN,NaN,NaN,sync check inconclusive (te_aligned qua nho)
4,b01,0.004474,116664,0.637953,PASS,0.126299,0.120772,972.0,NaN
5,c01,0.014724,115637,0.420422,PASS,0.246223,0.163644,963.0,NaN
6,c02,0.004475,119139,0.984762,FAIL,NaN,NaN,NaN,sync check khong dat (ratio=0.98)
7,c03,0.076415,108766,0.238967,PASS,0.204279,0.171923,906.0,NaN


## 4. ✅ KET LUAN — kiem dinh chinh thuc theo don vi ban ghi

Chi 8 ban ghi (co the it hon sau kiem tra dong bo) - suc manh thong ke han che. Dung lam **kiem tra huong nhat quan voi Fantasia**, khong ky vong CI hep.

In [5]:
passed = summary[summary['verdict'] == 'PASS']
print(f'{len(passed)}/{len(RECORDS_WITH_RESP)} ban ghi Apnea-ECG PASS.')

if len(passed) < 3:
    print('QUA IT ban ghi PASS de kiem dinh co y nghia - bao cao nhu mot quan sat, khong ket luan.')
else:
    fwd = passed['te_resp_to_rr'].tolist()
    bwd = passed['te_rr_to_resp'].tolist()
    res = run_sanity_check_per_record(fwd, bwd, n_bootstrap=cfg['n_bootstrap'], seed=cfg['seed'])
    print(f"TE(resp->tim)={res['te_forward_mean']:.4f}, TE(tim->resp)={res['te_backward_mean']:.4f}")
    print(f"CI hieu 95%: [{res['ci_difference'][0]:.4f}, {res['ci_difference'][1]:.4f}]")
    print(f"Wilcoxon p={res['wilcoxon_p']:.4f}")
    n_concordant = sum(1 for f, b in zip(fwd, bwd) if f > b)
    print(f"So ban ghi dung huong RSA (resp->tim > tim->resp): {n_concordant}/{len(passed)}")
    print()
    print('=== KET LUAN APNEA-ECG: ' + ('DUNG HUONG, KHOP VOI FANTASIA' if res['difference'] > 0 else 'NGUOC HUONG - CAN XEM LAI') + ' ===')

6/8 ban ghi Apnea-ECG PASS.
Sanity check (theo ban ghi, N=6) FAILED.
TE(resp->tim)=0.1362, TE(tim->resp)=0.1185
CI hieu 95%: [-0.0038, 0.0455]
Wilcoxon p=0.2188
So ban ghi dung huong RSA (resp->tim > tim->resp): 4/6

=== KET LUAN APNEA-ECG: DUNG HUONG, KHOP VOI FANTASIA ===


## 5. ✅ Kiem dinh GOP CA 2 BO DU LIEU (Fantasia + Apnea-ECG) - ket luan manh nhat

Apnea-ECG mot minh (N=6) khong du manh de co y nghia thong ke rieng. Nhung day la
**doc lap thong ke thuc su** voi Fantasia (nguon du lieu khac, doi tuong khac) -
gop ca 2 bo lam **1 kiem dinh chung tren N=23 ban ghi doc lap** la cach dung de tong
hop bang chung tu nhieu nguon (tuong tu meta-analysis), manh hon nhieu so voi xem
rieng tung bo.

In [ ]:
from pqrst.data.synthetic.corpus import load_corpus
from pqrst.evaluation.sanity_check import bidirectional_te_per_record

# Nap lai cua so Fantasia da luu boi notebook 02c (khong chay lai tu dau)
fantasia_dir = BASE / 'data' / 'processed' / 'fantasia'
fantasia_quality = pd.read_csv(fantasia_dir / 'quality_report.csv')
fantasia_passed = fantasia_quality.loc[fantasia_quality['verdict'] == 'PASS', 'record_id'].tolist()

records_fantasia = {}
for rid in fantasia_passed:
    w_rr_to_resp = load_corpus(str(fantasia_dir / (rid + '_fwd.npz')))
    w_resp_to_rr = load_corpus(str(fantasia_dir / (rid + '_bwd.npz')))
    records_fantasia[f'fantasia_{rid}'] = (w_resp_to_rr, w_rr_to_resp)

per_record_fantasia = bidirectional_te_per_record(records_fantasia, est_ksg)

# Ghep voi cac ban ghi Apnea-ECG da PASS o Muc 3-4 (bien `passed` co san trong kernel)
per_record_apnea = {
    f"apnea_{row['record_id']}": {'te_forward': row['te_resp_to_rr'], 'te_backward': row['te_rr_to_resp']}
    for _, row in passed.iterrows()
}

combined = {**per_record_fantasia, **per_record_apnea}
fwd_all = [v['te_forward'] for v in combined.values()]
bwd_all = [v['te_backward'] for v in combined.values()]

res_combined = run_sanity_check_per_record(fwd_all, bwd_all, n_bootstrap=cfg['n_bootstrap'], seed=cfg['seed'])
n_concordant_all = sum(1 for f, b in zip(fwd_all, bwd_all) if f > b)

print(f"GOP CA 2 BO (N={res_combined['n_records']} = {len(per_record_fantasia)} Fantasia + {len(per_record_apnea)} Apnea-ECG):")
print(f"  TE(resp->tim)={res_combined['te_forward_mean']:.4f}, TE(tim->resp)={res_combined['te_backward_mean']:.4f}")
print(f"  CI hieu 95%: [{res_combined['ci_difference'][0]:.4f}, {res_combined['ci_difference'][1]:.4f}]")
print(f"  Wilcoxon p={res_combined['wilcoxon_p']:.4f}")
print(f"  So ban ghi dung huong (ca 2 bo): {n_concordant_all}/{len(fwd_all)}")
print()
print('=== KET LUAN GOP: ' + ('DAT - bang chung manh nhat, doc lap 2 nguon du lieu' if res_combined['passed'] else 'CHUA DAT') + ' ===')
